# EWSR1-FLI1 CT Genes — HPA Tissue Expression

Reproductive tissue vs other expression of CT genes from Table 2 of:
> *EWSR1-FLI1 Activation of the Cancer/Testis Antigen FATE1 Promotes Ewing Sarcoma Survival*

Data source: Human Protein Atlas v23 (RNA tissue consensus + normal tissue IHC)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pirlygenes.gene_sets_cancer import CTA_gene_names

HPA_DATA = "/Users/iskander/code/experiments/nutm1-ctas/data/hpa"
RNA_PATH = f"{HPA_DATA}/v23_rna_tissue_consensus.tsv.zip"
PROTEIN_PATH = f"{HPA_DATA}/v23_normal_tissue.tsv.zip"
BULK_PATH = f"{HPA_DATA}/proteinatlas.tsv.zip"

In [ ]:
TABLE2_GENES = [
    "ACTL8", "ADAM2", "ADAM29", "ANKRD45", "ARX", "CABYR", "CCDC33",
    "CEP290", "CTCFL", "CTNNA2", "DDX53", "DPPA2", "ELOVL4", "FATE1",
    "FTHL17", "GPATCH2", "HORMAD2", "LEMD1", "LUZP4", "MAGEA1", "MAGEA4",
    "MORC1", "NOL4", "ODF1", "PRSS54", "SAGE1", "SLCO6A1", "SPACA3",
    "SPAG17", "SPATA19", "SYCP1", "TDRD1", "TEKT5", "TEX15", "TFDP3",
    "TMEFF1", "TMEFF2", "ZNF645", "NUF2", "OIP5",
]

CORE_REPRODUCTIVE = {"ovary", "placenta", "testis"}
EXTENDED_REPRODUCTIVE = CORE_REPRODUCTIVE | {
    "cervix", "endometrium", "endometrium 1", "endometrium 2",
    "epididymis", "fallopian tube", "prostate", "seminal vesicle", "vagina",
}
# Exclude thymus from denominator — AIRE-driven mTEC expression is expected for CTAs
EXCLUDED_TISSUES = {"thymus"}

pirlygenes_ctas = CTA_gene_names()
in_cta_set = {g for g in TABLE2_GENES if g in pirlygenes_ctas}
not_in_cta_set = {g for g in TABLE2_GENES if g not in pirlygenes_ctas}
print(f"{len(in_cta_set)} in pirlygenes CTA set, {len(not_in_cta_set)} not")

## Load HPA data

In [ ]:
rna_all = pd.read_csv(RNA_PATH, sep="\t", low_memory=False)
protein_all = pd.read_csv(PROTEIN_PATH, sep="\t", low_memory=False)
bulk_all = pd.read_csv(
    BULK_PATH, sep="\t", low_memory=False,
    usecols=["Gene", "Gene synonym", "Ensembl",
             "RNA tissue specificity", "RNA tissue distribution",
             "RNA tissue specific nTPM"],
)

# Filter to our genes
gene_set = set(TABLE2_GENES)
rna = rna_all[rna_all["Gene name"].isin(gene_set)].copy()
protein = protein_all[protein_all["Gene name"].isin(gene_set)].copy()
bulk = bulk_all[bulk_all["Gene synonym"].fillna("").apply(
    lambda x: bool(gene_set & set(x.split(", ")))
) | bulk_all["Gene"].isin(gene_set)].copy()

print(f"RNA rows: {len(rna)} covering {rna['Gene name'].nunique()} genes")
print(f"Protein rows: {len(protein)} covering {protein['Gene name'].nunique()} genes")
print(f"Bulk rows: {len(bulk)}")

# Genes not found in HPA
found_rna = set(rna["Gene name"].unique())
found_protein = set(protein["Gene name"].unique())
missing = gene_set - (found_rna | found_protein)
if missing:
    print(f"\nNot found in HPA: {sorted(missing)}")

## RNA expression by tissue

In [ ]:
def classify_tissue(tissue: str) -> str:
    t = tissue.lower().strip()
    if t in CORE_REPRODUCTIVE:
        return "core reproductive"
    elif t in EXTENDED_REPRODUCTIVE:
        return "extended reproductive"
    else:
        return "other"


rna["tissue_class"] = rna["Tissue"].map(classify_tissue)

# Summarize per gene: max nTPM in reproductive vs other
rna_summary = []
for gene, gdf in rna.groupby("Gene name"):
    core = gdf[gdf["tissue_class"] == "core reproductive"]
    extended = gdf[gdf["tissue_class"].isin(["core reproductive", "extended reproductive"])]
    other = gdf[gdf["tissue_class"] == "other"]
    detected = gdf[gdf["nTPM"] >= 1.0]
    repro_detected = detected[detected["tissue_class"] != "other"]
    other_detected = detected[detected["tissue_class"] == "other"]

    total_ntpm = gdf["nTPM"].sum()
    core_ntpm = core["nTPM"].sum()
    repro_frac = core_ntpm / total_ntpm if total_ntpm > 0 else np.nan

    rna_summary.append({
        "gene": gene,
        "in_pirlygenes_cta": gene in pirlygenes_ctas,
        "max_nTPM_core_repro": core["nTPM"].max() if len(core) else 0,
        "max_nTPM_other": other["nTPM"].max() if len(other) else 0,
        "sum_nTPM_total": total_ntpm,
        "sum_nTPM_core_repro": core_ntpm,
        "core_repro_fraction": repro_frac,
        "n_tissues_detected": len(detected),
        "n_repro_tissues_detected": len(repro_detected),
        "n_other_tissues_detected": len(other_detected),
        "detected_other_tissues": "; ".join(sorted(other_detected["Tissue"].unique())),
        "top_tissue": gdf.loc[gdf["nTPM"].idxmax(), "Tissue"] if len(gdf) else "",
        "top_nTPM": gdf["nTPM"].max() if len(gdf) else 0,
    })

rna_summary_df = pd.DataFrame(rna_summary).sort_values("core_repro_fraction", ascending=False)
rna_summary_df

## Protein expression by tissue

In [ ]:
protein["tissue_class"] = protein["Tissue"].map(classify_tissue)
protein_detected = protein[protein["Level"].astype(str) != "Not detected"].copy()

protein_summary = []
for gene, gdf in protein_detected.groupby("Gene name"):
    core = gdf[gdf["tissue_class"] == "core reproductive"]
    other = gdf[gdf["tissue_class"] == "other"]
    all_tissues = set(gdf["Tissue"].str.lower().unique())
    core_restricted = all_tissues.issubset(CORE_REPRODUCTIVE) and bool(all_tissues)
    ext_restricted = all_tissues.issubset(EXTENDED_REPRODUCTIVE) and bool(all_tissues)

    protein_summary.append({
        "gene": gene,
        "in_pirlygenes_cta": gene in pirlygenes_ctas,
        "n_tissues_detected": gdf["Tissue"].nunique(),
        "n_core_repro_tissues": core["Tissue"].nunique(),
        "n_other_tissues": other["Tissue"].nunique(),
        "core_restricted": core_restricted,
        "extended_restricted": ext_restricted,
        "max_level_core_repro": core["Level"].map({"Low": 1, "Medium": 2, "High": 3}).max() if len(core) else 0,
        "max_level_other": other["Level"].map({"Low": 1, "Medium": 2, "High": 3}).max() if len(other) else 0,
        "detected_tissues": "; ".join(sorted(gdf["Tissue"].unique())),
        "detected_other_tissues": "; ".join(sorted(other["Tissue"].unique())),
        "levels": "; ".join(sorted(gdf["Level"].unique())),
        "reliability": "; ".join(sorted(gdf["Reliability"].dropna().unique())),
    })

protein_summary_df = pd.DataFrame(protein_summary).sort_values(
    ["core_restricted", "n_other_tissues"], ascending=[False, True]
)

# Also note genes with no protein evidence at all
no_protein = gene_set - set(protein_detected["Gene name"].unique())
print(f"Genes with no protein detection in any tissue: {sorted(no_protein)}\n")
protein_summary_df

## Combined summary

In [ ]:
combined = (
    rna_summary_df[["gene", "in_pirlygenes_cta", "core_repro_fraction",
                     "n_tissues_detected", "n_other_tissues_detected",
                     "max_nTPM_core_repro", "max_nTPM_other", "top_tissue"]]
    .rename(columns=lambda c: f"rna_{c}" if c not in ("gene", "in_pirlygenes_cta") else c)
    .merge(
        protein_summary_df[["gene", "n_tissues_detected", "n_other_tissues",
                            "core_restricted", "extended_restricted",
                            "detected_other_tissues"]]
        .rename(columns=lambda c: f"protein_{c}" if c != "gene" else c),
        on="gene", how="outer",
    )
)

combined = combined.sort_values("rna_core_repro_fraction", ascending=False)
combined

## RNA heatmap — nTPM by tissue

In [ ]:
# Pivot RNA data into gene x tissue matrix
rna_pivot = rna.pivot_table(index="Gene name", columns="Tissue", values="nTPM", aggfunc="max")

# Order genes by reproductive fraction
gene_order = rna_summary_df.sort_values("core_repro_fraction", ascending=True)["gene"].tolist()
gene_order = [g for g in gene_order if g in rna_pivot.index]
rna_pivot = rna_pivot.loc[gene_order]

# Order tissues: reproductive first, then other
all_tissues = sorted(rna_pivot.columns)
repro_tissues = [t for t in all_tissues if t.lower() in EXTENDED_REPRODUCTIVE]
other_tissues = [t for t in all_tissues if t.lower() not in EXTENDED_REPRODUCTIVE]
tissue_order = repro_tissues + other_tissues
rna_pivot = rna_pivot[tissue_order]

fig, ax = plt.subplots(figsize=(22, 12))
sns.heatmap(
    np.log1p(rna_pivot.fillna(0)),
    cmap="YlOrRd", ax=ax, linewidths=0.3, linecolor="#eeeeee",
    cbar_kws={"label": "log(1 + nTPM)", "shrink": 0.6},
)

# Mark the divider between reproductive and other tissues
ax.axvline(x=len(repro_tissues), color="black", linewidth=2)
ax.set_title("RNA expression (nTPM) — EWSR1-FLI1 CT genes\n"
             f"Left of black line: reproductive tissues | Right: other tissues",
             fontsize=14, fontweight="bold", loc="left")
ax.set_ylabel("")
ax.set_xlabel("")

# Highlight genes in pirlygenes CTA set
for i, label in enumerate(ax.get_yticklabels()):
    if label.get_text() in pirlygenes_ctas:
        label.set_fontweight("bold")
        label.set_color("#1a6600")

plt.tight_layout()
plt.show()

## Protein detection heatmap

In [ ]:
level_map = {"Not detected": 0, "Low": 1, "Medium": 2, "High": 3}
protein["level_numeric"] = protein["Level"].map(level_map)

# Pivot: max level per gene x tissue
prot_pivot = protein.pivot_table(
    index="Gene name", columns="Tissue", values="level_numeric", aggfunc="max"
)

# Order genes same as RNA
gene_order_prot = [g for g in gene_order if g in prot_pivot.index]
# Add genes only in protein data
gene_order_prot += [g for g in prot_pivot.index if g not in gene_order_prot]
prot_pivot = prot_pivot.loc[gene_order_prot]

# Order tissues: reproductive first
all_prot_tissues = sorted(prot_pivot.columns)
repro_prot = [t for t in all_prot_tissues if t.lower() in EXTENDED_REPRODUCTIVE]
other_prot = [t for t in all_prot_tissues if t.lower() not in EXTENDED_REPRODUCTIVE]
prot_pivot = prot_pivot[repro_prot + other_prot]

fig, ax = plt.subplots(figsize=(24, 12))
cmap = sns.color_palette(["#f0f0f0", "#fdd49e", "#fc8d59", "#b30000"], as_cmap=True)
sns.heatmap(
    prot_pivot.fillna(0), cmap=cmap, ax=ax, linewidths=0.3, linecolor="#eeeeee",
    vmin=0, vmax=3,
    cbar_kws={"label": "Protein level (0=ND, 1=Low, 2=Med, 3=High)", "shrink": 0.6},
)
ax.axvline(x=len(repro_prot), color="black", linewidth=2)
ax.set_title("Protein expression (IHC) — EWSR1-FLI1 CT genes\n"
             f"Left of black line: reproductive tissues | Right: other tissues",
             fontsize=14, fontweight="bold", loc="left")
ax.set_ylabel("")
ax.set_xlabel("")

for i, label in enumerate(ax.get_yticklabels()):
    if label.get_text() in pirlygenes_ctas:
        label.set_fontweight("bold")
        label.set_color("#1a6600")

plt.tight_layout()
plt.show()

## Reproductive restriction classification

In [ ]:
# Classify each gene
# RNA restriction: >=80% of total nTPM (excluding thymus) in core reproductive tissues
RNA_REPRO_THRESHOLD = 0.80

classification = []
for gene in TABLE2_GENES:
    row = {"gene": gene, "in_pirlygenes_cta": gene in pirlygenes_ctas}

    # RNA — exclude thymus from the denominator
    g_rna = rna[rna["Gene name"] == gene]
    g_rna_no_thymus = g_rna[~g_rna["Tissue"].str.lower().isin(EXCLUDED_TISSUES)]
    rna_detected = g_rna_no_thymus[g_rna_no_thymus["nTPM"] >= 1.0]
    rna_tissues = set(rna_detected["Tissue"].str.lower())
    row["rna_detected_n"] = len(rna_tissues)
    total = g_rna_no_thymus["nTPM"].sum()
    core_sum = g_rna_no_thymus[g_rna_no_thymus["Tissue"].str.lower().isin(CORE_REPRODUCTIVE)]["nTPM"].sum()
    row["rna_core_fraction"] = round(core_sum / total, 3) if total > 0 else np.nan
    row["rna_core_restricted"] = (
        not np.isnan(row["rna_core_fraction"]) and row["rna_core_fraction"] >= RNA_REPRO_THRESHOLD
    )
    # Note thymus expression separately
    thymus_ntpm = g_rna[g_rna["Tissue"].str.lower() == "thymus"]["nTPM"].sum()
    row["thymus_nTPM"] = thymus_ntpm

    # Protein (strict subset — no continuous measure for IHC levels)
    g_prot = protein_detected[protein_detected["Gene name"] == gene]
    prot_tissues = set(g_prot["Tissue"].dropna().str.lower()) - EXCLUDED_TISSUES
    row["protein_detected_n"] = len(prot_tissues)
    row["protein_core_restricted"] = prot_tissues.issubset(CORE_REPRODUCTIVE) and bool(prot_tissues)
    row["protein_extended_restricted"] = prot_tissues.issubset(EXTENDED_REPRODUCTIVE) and bool(prot_tissues)
    row["protein_detected_tissues"] = "; ".join(sorted(prot_tissues)) if prot_tissues else "none"

    # Overall
    if row["rna_core_restricted"] and row["protein_core_restricted"]:
        row["verdict"] = "CORE RESTRICTED (both)"
    elif row["rna_core_restricted"] or row["protein_core_restricted"]:
        row["verdict"] = "CORE RESTRICTED (one modality)"
    elif row["protein_extended_restricted"]:
        row["verdict"] = "EXTENDED RESTRICTED (protein)"
    elif row["rna_detected_n"] == 0 and row["protein_detected_n"] == 0:
        row["verdict"] = "NO HPA DATA"
    else:
        row["verdict"] = "BROADLY EXPRESSED"

    classification.append(row)

class_df = pd.DataFrame(classification)
print(f"RNA restriction: >={RNA_REPRO_THRESHOLD:.0%} of nTPM in core reproductive tissues (thymus excluded)")
print(f"Protein restriction: all detected tissues (thymus excluded) in core reproductive set\n")
print(class_df["verdict"].value_counts().to_string())
print()
class_df.sort_values(["verdict", "gene"])

In [ ]:
# HPA bulk summary annotations for these genes
bulk_for_genes = bulk_all[bulk_all["Gene"].isin(gene_set)][
    ["Gene", "RNA tissue specificity", "RNA tissue distribution", "RNA tissue specific nTPM"]
].copy()
bulk_for_genes = bulk_for_genes.sort_values("Gene")
bulk_for_genes